Librarii

In [17]:
# Import required libraries
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm


1. Data Loading
* dataset1: 3 foldere normal/malign/benign, cu imagini
* dataset2: Isoric al pacientilor
* dataset3: Metadata

Clasa Dataset pt a lua datele

Am separat datele in malign sau benign/normal

In [18]:
class MultiFolderMultimodalDataset(Dataset):

  def __init__(self, base_dir, transform=None):
    self.base_dir = base_dir
    self.transform = transform


    self.patient_df = pd.read_csv(
        os.path.join(base_dir, "dataset2", "patient_history_dataset.csv")
    )
    self.biomarker_df = pd.read_csv(
        os.path.join(base_dir, "dataset3", "molecular_biomarker_dataset.csv")
    )

    #merge dataset 2 and 3
    self.metadata = pd.merge(
        self.patient_df, self.biomarker_df, on="Patient ID", how="inner"
    )

    self.samples = []
    classes = ["normal", "benign", "malignant"]
    for cls in classes:
      img_folder = os.path.join(base_dir, "dataset1", cls, "images")
      if os.path.exists(img_folder):
        for img_name in os.listdir(img_folder):
          if img_name.lower().endswith((".png", ".jpg", ".jpeg", ".tif")):
            self.samples.append({
                "img_path": os.path.join(img_folder, img_name),
                "class_name": cls,
            })

    self.samples_df = pd.DataFrame(self.samples)

  def __len__(self):
    return len(self.samples_df)

  def __getitem__(self, idx):
    row = self.samples_df.iloc[idx]

    # Load and transform image
    image = Image.open(row["img_path"]).convert("RGB")
    if self.transform:
      image = self.transform(image)

    # Extract numerical features using exact CSV column names
    tabular_tensor = torch.tensor(
        [
            float(self.metadata.iloc[idx].get("Age at Diagnosis", 0.0)),
            float(self.metadata.iloc[idx].get("Neoplasm Histologic Grade", 0.0)),
        ],
        dtype=torch.float32,
    )

    # malign = 1, normal/benign = 0
    label_val = 1.0 if row["class_name"] == "malignant" else 0.0
    label = torch.tensor(label_val, dtype=torch.float32)

    return image, tabular_tensor, label

2. Multi-Modal Fusion
* Image Branch: Foloseste EfficientNet preantrenat (timm) pentru a extrage embeddings
* Tabular Branch: Foloseste un Multi-Layer Perceptron (MLP) pentru a procesa istoric si biomarker data
* Fusion Layer: Concateneaza embeddingurile

In [13]:
class MultimodalFusionModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_tabular_features=2):
        super(MultimodalFusionModel, self).__init__()

        # 1. Image Encoder
        self.image_encoder = timm.create_model(model_name, pretrained=True, num_classes=0)
        img_feature_dim = self.image_encoder.num_features

        # 2. Tabular branch
        self.tabular_encoder = nn.Sequential(
            nn.Linear(num_tabular_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # 3. Fusion and Classifier
        fusion_input_dim = img_feature_dim + 16
        self.classifier = nn.Sequential(
            nn.Linear(fusion_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, images, tabular):
        img_feats = self.image_encoder(images)
        tab_feats = self.tabular_encoder(tabular)

        combined = torch.cat((img_feats, tab_feats), dim=1)
        out = self.classifier(combined)
        return out.squeeze(1)

3. Training


4. Separare Test and Train Data

In [15]:
from torch.utils.data import DataLoader, random_split
from torchvision import transforms

# Transformari
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

base_dir = "../data/dataset/"
full_dataset = MultiFolderMultimodalDataset(base_dir=base_dir, transform=transform)

#  80% train, 20% test
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size]
)

print(f"Total samples: {len(full_dataset)}")
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

Total samples: 780
Training samples: 624
Test samples: 156


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

#
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalFusionModel(
    model_name="efficientnet_b0", num_tabular_features=2
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 3

for epoch in range(epochs):
  print(f"\n--- Epoch {epoch+1}/{epochs} ---")

  # --- TRAINING PHASE ---
  model.train()
  running_loss = 0.0

  for images, tabular, labels in train_loader:
    images = images.to(device)
    tabular = tabular.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images, tabular)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

  epoch_train_loss = running_loss / len(train_loader.dataset)
  print(f"Train Loss: {epoch_train_loss:.4f}")

  # --- EVALUATION / TESTING PHASE ---
  model.eval()
  correct = 0
  total = 0
  test_loss = 0.0

  with torch.no_grad():
    for images, tabular, labels in test_loader:
      images = images.to(device)
      tabular = tabular.to(device)
      labels = labels.to(device)

      outputs = model(images, tabular)
      loss = criterion(outputs, labels)
      test_loss += loss.item() * images.size(0)


      probs = torch.sigmoid(outputs)
      predicted = (probs >= 0.5).float()

      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  epoch_test_loss = test_loss / len(test_loader.dataset)
  test_accuracy = (correct / total) * 100 if total > 0 else 0

  print(
      f"Test Loss: {epoch_test_loss:.4f} | Test Accuracy: {test_accuracy:.2f}%"
  )


--- Epoch 1/3 ---
Train Loss: 0.5776
Test Loss: 0.4771 | Test Accuracy: 75.64%

--- Epoch 2/3 ---
Train Loss: 0.2987
Test Loss: 0.3309 | Test Accuracy: 85.90%

--- Epoch 3/3 ---
Train Loss: 0.1259
Test Loss: 0.2525 | Test Accuracy: 91.03%
